# **Cyber Physical Safety "Anomaly Detection" Project**
Shamma Almheiri

ID : 10424539

In [ ]:
import os
import sys

# Set the paths for a local Mac environment
os.environ['JAVA_HOME'] = sys.prefix
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession

# Lowering memory to 2g to prevent Mac crashes
spark = SparkSession.builder \
    .appName("Aviation_Local_Fix") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark is finally running!")


In [ ]:
#Initialize Spark - Architectural Setup
from pyspark.sql import SparkSession

# We allocate 4GB of memory to handle the large CSV file from Drive
spark = SparkSession.builder \
    .appName("Aviation_Cyber_Security") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()

print("Spark Session Active and Configured.")

In [ ]:
# Ingest Data from Drive - Task 1 - Ingestion

file_path = "/content/drive/MyDrive/flightlist.csv"

# Using 'inferSchema=True' to let Spark detect data types automatically
df_raw = spark.read.csv(file_path, header=True, inferSchema=True)

# 1. Verification of Volume (Required for 1M+ records)
record_count = df_raw.count()
print(f"Total Records Ingested from Drive: {record_count:,}")

# 2. Show the structure
df_raw.printSchema()

In [ ]:
# Partitioning Strategy
# We re-partition the data into 20 chunks based on the Aircraft ID.
# This prevents 'Data Skew' (one worker doing all the work).
df_partitioned = df_raw.repartition(20, "icao24")

print("Architecture Optimized: Data partitioned by Aircraft ID (icao24).")


In [ ]:
# Task 2 - Handling Missing Values

from pyspark.sql.functions import col, when, rand, lit, abs

# 1. DROP rows that are missing critical geographical data
# We can't detect anomalies if we don't have the coordinates.
df_clean = df_partitioned.dropna(subset=["latitude_1", "longitude_1", "altitude_1", "latitude_2", "longitude_2", "altitude_2"])

# 2. FILL missing callsigns with "UNKNOWN" (Categorical handling)
df_clean = df_clean.fillna({"callsign": "UNKNOWN", "typecode": "NONE"})

print(f"Cleaning complete. Records remaining: {df_clean.count():,}")

In [ ]:
# Advanced Feature Engineering

# Create a feature showing how much the altitude changed during the flight segment
df_features = df_clean.withColumn("alt_change", abs(col("altitude_2") - col("altitude_1")))

# Create a feature for Latitude/Longitude displacement (Simplified distance)
df_features = df_features.withColumn("lat_dist", abs(col("latitude_2") - col("latitude_1")))
df_features = df_features.withColumn("lon_dist", abs(col("longitude_2") - col("longitude_1")))

print("Feature Engineering complete: Calculated Vertical and Horizontal displacement.")

In [ ]:
# Anomaly Injection (Creating our "Labels")

# 1. Initialize all records as 'Normal' (0)
df_labeled = df_features.withColumn("label", lit(0))

# 2. Inject Anomaly Logic:
# We simulate a "GPS Spoofing" attack by making the altitude or coordinates physically impossible
# For 10% of the data, we multiply the altitude by 10 or offset the latitude significantly.

df_final = df_labeled.withColumn("label",
    when(rand(seed=42) < 0.10, 1).otherwise(0)
).withColumn("altitude_1",
    when(col("label") == 1, col("altitude_1") * 10).otherwise(col("altitude_1"))
).withColumn("latitude_1",
    when(col("label") == 1, col("latitude_1") + 5.0).otherwise(col("latitude_1"))
)

print("Labels Created: 0 = Normal, 1 = Spoofing Attack")
df_final.groupBy("label").count().show()

In [ ]:
# Normalization & Vectorization (preperation for task 3)

from pyspark.ml.feature import VectorAssembler, StandardScaler

# Define the features we want the AI to look at
feature_cols = ["latitude_1", "longitude_1", "altitude_1", "alt_change", "lat_dist", "lon_dist"]

# Assemble them into a single column named 'features'
assembler = VectorAssembler(inputCols=feature_cols, outputCol="unscaled_features")
df_vector = assembler.transform(df_final)

# Normalize the data (Scale it so all values are between 0 and 1)
# This is critical for Neural Network convergence
scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_vector)
df_model_ready = scaler_model.transform(df_vector)

# Select only the features and the label for the model
df_ml = df_model_ready.select("features", "label")

print("Data is now Scaled, Vectorized, and Ready for the Neural Network!")
df_ml.show(5)

In [ ]:
# Phase B
# Scalable Deep Learning

# Split the Data - We need to hide some data from the AI so we can test its accuracy later

# Split the data into 80% for training and 20% for testing
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Training set size: {train_df.count():,}")
print(f"Testing set size: {test_df.count():,}")

In [ ]:
# Define and Train the Neural Network (MLP)

from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# This is "Brain-inspired" because it has multiple hidden layers.
layers = [6, 16, 8, 2]

# Initialize the Neural Network
# We set maxIter (iterations) and blockSize (batch size)
mlp = MultilayerPerceptronClassifier(
    layers=layers,
    blockSize=128,
    seed=1234,
    maxIter=100,
    labelCol="label",
    featuresCol="features")

# Train the model (This might take a minute because of the large data)
print("Training the Deep Neural Network... processing 2 million records.")
model = mlp.fit(train_df)

print("Training Complete!")

In [ ]:
# Test Accuracy & Hyperparameter Tuning
# We need to prove we hit the 80% accuracy target.

# Make predictions on the test data
predictions = model.transform(test_df)

# Evaluate Accuracy
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print(f"Model Accuracy: {accuracy * 100:.2f}%")


In [ ]:
# Hyperparameter Tuning

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Create a grid of parameters to test
# 1. We test different Step Sizes (Learning Rate)
# 2. We test different Max Iterations
paramGrid = ParamGridBuilder() \
    .addGrid(mlp.stepSize, [0.01, 0.1]) \
    .addGrid(mlp.maxIter, [50, 100]) \
    .build()

# Cross-validation (Checks if the model works on different data slices)
crossval = CrossValidator(estimator=mlp,
                          estimatorParamMaps=paramGrid,
                          evaluator=evaluator,
                          numFolds=2) # 2 folds to keep it fast for Colab

print("Optimizing hyperparameters to find the best configuration...")
cv_model = crossval.fit(train_df)
print("Hyperparameter Tuning Complete.")

In [ ]:
# Performance Benchmarking (Scalability Analysis)

import time
import matplotlib.pyplot as plt

# 1. Define different data scales (20%, 50%, 100%)
scales = [0.2, 0.5, 1.0]
execution_times = []

print("Starting Scalability Benchmarking...")

for scale in scales:
    start_time = time.time()

    # Perform a heavy transformation on a fraction of the data
    temp_df = df_ml.sample(False, scale, seed=42)
    # We use .count() to force Spark to execute the plan (Action)
    count = temp_df.count()

    end_time = time.time()
    duration = end_time - start_time
    execution_times.append(duration)
    print(f"Scale {int(scale*100)}%: Processed {count:,} records in {duration:.2f} seconds")

# 2. Plot the Scalability Graph (Required for the Report)
plt.figure(figsize=(10, 6))
plt.plot([s * 100 for s in scales], execution_times, marker='o', linestyle='-', color='b')
plt.title('Scalability Analysis: Execution Time vs. Data Scale')
plt.xlabel('Data Volume (%)')
plt.ylabel('Execution Time (Seconds)')
plt.grid(True)
plt.show()

In [ ]:
# XAI Validation (The "Glass Box" Logic)

import matplotlib.pyplot as plt
import numpy as np

# 1. Access the trained Neural Network weights
# These are the actual 'importance' values learned by the 2.6M records
mlp_weights = model.weights.toArray()

# 2. Extract the weights for our 6 input features
# We take the absolute mean of the weights connected to the first layer
# This shows how much 'attention' the brain-inspired model pays to each feature.
raw_importance = [np.abs(mlp_weights[i]) for i in range(6)]

# 3. Create the bar chart for your report
feature_names = ["Lat_1", "Lon_1", "Alt_1", "Alt_Change", "Lat_Dist", "Lon_Dist"]

plt.figure(figsize=(10, 6))
plt.bar(feature_names, raw_importance, color='darkorchid')
plt.title('XAI Validation: Neural Network Feature Importance (Glass Box Logic)')
plt.ylabel('Learned Weight Magnitude')
plt.xlabel('Aviation Signal Features')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("XAI Feature Importance Plot generated successfully!")

In [ ]:
# Phase D
# The AI GUI
import gradio as gr

# 1. Prediction function for the Gradio Interface
def validate_adsb_signal(lat, lon, alt, alt_chg, lat_d, lon_d):
    try:
        # Prepare the input row for Spark
        data = [(float(lat), float(lon), float(alt), float(alt_chg), float(lat_d), float(lon_d))]
        cols = ["latitude_1", "longitude_1", "altitude_1", "alt_change", "lat_dist", "lon_dist"]

        input_df = spark.createDataFrame(data, cols)

        # Run through the Spark Pipeline
        vec_df = assembler.transform(input_df)
        scaled_df = scaler_model.transform(vec_df)

        # Get the prediction (0 = Normal, 1 = Attack)
        prediction = model.transform(scaled_df).select("prediction").collect()[0][0]

        # Logic for professional output
        if prediction == 1:
            result = "🛑 CYBER-ATTACK DETECTED"
            explanation = "XAI Logic: The model flagged this signal as 'Spoofed' due to physically impossible shifts in Longitude/Latitude or Altitude."
        else:
            result = "✅ SIGNAL SECURE"
            explanation = "XAI Logic: The flight parameters align with standard aviation physics and trajectories."

        return result, explanation
    except Exception as e:
        return "Error", str(e)

# 2. Build the 2-Tab Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Emirates Aviation: Cyber-Physical Safety Monitor")
    gr.Markdown("Real-time ADS-B Signal Anomaly Detection using Distributed Deep Learning.")

    with gr.Tab("Validator"):
        gr.Markdown("### Enter Signal Parameters")
        with gr.Row():
            lat_in = gr.Number(label="Latitude", value=25.2)
            lon_in = gr.Number(label="Longitude", value=55.3)
            alt_in = gr.Number(label="Altitude (ft)", value=35000)
        with gr.Row():
            alt_c = gr.Number(label="Alt Change", value=100)
            lat_d = gr.Number(label="Lat Displacement", value=0.01)
            lon_d = gr.Number(label="Lon Displacement", value=0.01)

        btn = gr.Button("Analyze Signal", variant="primary")
        out_status = gr.Textbox(label="Detection Result")
        out_xai = gr.Textbox(label="Model Explanation (XAI)")

        btn.click(fn=validate_adsb_signal,
                  inputs=[lat_in, lon_in, alt_in, alt_c, lat_d, lon_d],
                  outputs=[out_status, out_xai])

    with gr.Tab("Architecture Insights"):
        gr.Markdown("### Senior Architect Summary")
        gr.Markdown(f"- **Dataset Size:** 2,669,349 records")
        gr.Markdown("- **Processing Engine:** Distributed PySpark (Lazy Evaluation)")
        gr.Markdown("- **Model Type:** Multi-Layer Perceptron (Deep Learning)")
        gr.Markdown("- **Validation Accuracy:** 95.19%")
        gr.Markdown("- **Interpretability:** Weight Magnitude Analysis (Glass Box Logic)")

demo.launch(share=True)

In [ ]:
# Extra Step

# I added this step, so it becomes easier for us to test the product

# 1. Get a RANDOM Normal signal from your 2.6 Million records
print("--- 🎲 RANDOM NORMAL SIGNAL (Input these for 'Secure') ---")
df_final.filter(col("label") == 0) \
        .sample(withReplacement=False, fraction=0.01) \
        .select("latitude_1", "longitude_1", "altitude_1", "alt_change", "lat_dist", "lon_dist") \
        .show(1)

# 2. Get a RANDOM Spoofed signal from your 2.6 Million records
print("--- 🎲 RANDOM SPOOFED SIGNAL (Input these for 'Attack') ---")
df_final.filter(col("label") == 1) \
        .sample(withReplacement=False, fraction=0.1) \
        .select("latitude_1", "longitude_1", "altitude_1", "alt_change", "lat_dist", "lon_dist") \
        .show(1)
